# 11.14 — Policy Gradients (REINFORCE)

Policy gradients learn a stochastic policy directly: instead of first learning a value table and then acting greedily, REINFORCE nudges the parameters that made sampled actions more likely when their later return was good, and less likely when their later return was bad. In this lesson, we build the softmax policy, the log-derivative trick, discounted returns, baselines, and a tiny from-scratch training loop with only NumPy and Matplotlib.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build REINFORCE one idea at a time. Run each cell in order and read the printed intermediate values — every piece of probability and gradient math is shown so the update rule is not a black box. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, probabilities, sampling, and vector arithmetic.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for sampled episodes and training curves.

### 1. Return: delayed reward is the learning signal

A policy-gradient agent does not learn from the immediate reward alone; it learns from the **return** $G_t=r_t+\gamma r_{t+1}+\gamma^2 r_{t+2}+\cdots$. The discount $\gamma$ keeps future reward important but not free. This matters because the action we update now may only reveal whether it was wise several steps later.

In [ ]:
rewards_w = np.array([1.0, 0.0, 2.0])  # reward stream from one short episode.
gamma_w = 0.9  # future rewards count, but each step away is discounted.
powers_w = gamma_w ** np.arange(len(rewards_w))  # [1, gamma, gamma^2].
print("discount powers:", np.round(powers_w, 3))  # inspect the weights.
print("weighted rewards:", np.round(powers_w * rewards_w, 3))  # each reward's contribution.

▶ What you'll see: the delayed reward of 2 still contributes, but two steps later it is weighted by 0.81.

In [ ]:
G0_w = float(np.sum(powers_w * rewards_w))  # three-step discounted return from t=0.
print("G0 =", round(G0_w, 3))  # 1 + 0.9*0 + 0.9^2*2 = 2.62.
assert round(G0_w, 3) == 2.620  # concrete check from the lesson content.

▶ What you'll see: the full return is 2.620, not the immediate reward 1.000.

In [ ]:
returns_w = []  # store G_t for each time t.
running_w = 0.0  # accumulator for the backward recurrence.
for r_w in rewards_w[::-1]:  # work backward so future return is already known.
    running_w = r_w + gamma_w * running_w  # G_t = r_t + gamma G_{t+1}.
    returns_w.append(running_w)  # append in reverse order for now.
returns_w = np.array(returns_w[::-1])  # restore chronological order.
print("returns G_t:", np.round(returns_w, 3))  # inspect all time-step targets.
plt.figure(figsize=(4.4, 3))
plt.bar(["G0", "G1", "G2"], returns_w, color="teal")
plt.title("1: discounted returns per step"); plt.ylabel("return"); plt.show()

▶ What you'll see: each earlier action receives credit for rewards that arrived later in the same episode.

*Why it's done this way:* REINFORCE is an episodic Monte Carlo method, so it waits until the episode reveals what actually happened and then uses $G_t$ as the consequence of action $a_t$. The backward recurrence is algebraically the same discounted sum, but it avoids recomputing a long sum from scratch at every time step.

### 2. A softmax policy: logits become action probabilities

A policy gradient needs a differentiable policy $\pi_\theta(a\mid s)$. For a small discrete action set, the simplest choice is a vector of logits $z_\theta(s)$ passed through softmax. A larger logit gets more probability, but every action keeps nonzero support, which preserves exploration.

In [ ]:
logits_w = np.array([1.0, 0.0])  # two action preferences before normalization.
shift_w = logits_w - np.max(logits_w)  # numerical stability: softmax is unchanged by shifting.
exp_w = np.exp(shift_w)  # positive unnormalized weights.
probs_w = exp_w / np.sum(exp_w)  # normalized probabilities.
print("exp weights:", np.round(exp_w, 3))
print("policy probs:", np.round(probs_w, 3))
assert np.allclose(np.round(probs_w, 3), [0.731, 0.269])

▶ What you'll see: logits `[1, 0]` become probabilities about `[0.731, 0.269]`.

In [ ]:
rewards_if_chosen_w = np.array([2.0, 0.0])  # reward attached to each action in a one-step bandit.
expected_reward_w = float(np.dot(probs_w, rewards_if_chosen_w))  # E[R] under the policy.
print("expected reward:", round(expected_reward_w, 3))  # 0.731*2 + 0.269*0.
assert round(expected_reward_w, 3) == 1.462

▶ What you'll see: probability mass on the rewarding action directly raises expected reward.

In [ ]:
temps_w = np.array([0.5, 1.0, 2.0])  # lower temperature is greedier; higher is more exploratory.
curves_w = []
for temp_w in temps_w:
    p_w = np.exp(logits_w / temp_w - np.max(logits_w / temp_w))
    curves_w.append(p_w / p_w.sum())
print("P(action0) by temperature:", np.round([p[0] for p in curves_w], 3))
plt.figure(figsize=(4.6, 3))
plt.bar(["T=0.5", "T=1", "T=2"], [p[0] for p in curves_w], color="darkorange")
plt.ylim(0, 1); plt.ylabel("P(action 0)"); plt.title("2: temperature changes exploration"); plt.show()

▶ What you'll see: the low-temperature policy is more decisive, while the high-temperature policy stays more exploratory.

*Why it's done this way:* softmax turns unconstrained parameters into valid probabilities and keeps the map smooth, so tiny logit changes create tiny probability changes. That smoothness is what makes gradient ascent possible; hard argmax would choose actions but would not give a usable derivative.

### 3. The log-derivative trick: differentiating through sampling

The objective is $J(\theta)=\mathbb{E}_{\tau\sim\pi_\theta}[G(\tau)]$. The sample itself is random, so we use the identity $\nabla_\theta p_\theta(a)=p_\theta(a)\nabla_\theta\log p_\theta(a)$. That converts a derivative of a probability into a score function we can evaluate on the sampled action.

In [ ]:
p_w = probs_w.copy()  # use the softmax probabilities from concept 2.
a_w = 0  # suppose action 0 was sampled.
onehot_w = np.array([1.0, 0.0])  # one-hot vector for action 0.
grad_logp_w = onehot_w - p_w  # gradient of log softmax wrt logits.
print("grad log pi(action 0):", np.round(grad_logp_w, 3))
assert np.allclose(np.round(grad_logp_w, 3), [0.269, -0.269])

▶ What you'll see: increasing the chosen action's logit raises its log-probability; increasing the other logit lowers it.

In [ ]:
a_w = 1  # now suppose action 1 was sampled instead.
onehot_w = np.array([0.0, 1.0])
grad_logp_alt_w = onehot_w - p_w
print("grad log pi(action 1):", np.round(grad_logp_alt_w, 3))

▶ What you'll see: the signs flip because the sampled action is now action 1.

In [ ]:
advantage_w = 2.62  # positive consequence says "make the sampled action more likely."
update0_w = advantage_w * grad_logp_w  # REINFORCE direction for action 0.
update1_w = advantage_w * grad_logp_alt_w  # REINFORCE direction for action 1.
print("positive-return update if a0 sampled:", np.round(update0_w, 3))
print("positive-return update if a1 sampled:", np.round(update1_w, 3))
plt.figure(figsize=(4.8, 3))
plt.bar(["logit0 if a0", "logit1 if a0", "logit0 if a1", "logit1 if a1"],
        [update0_w[0], update0_w[1], update1_w[0], update1_w[1]], color=["green", "red", "red", "green"])
plt.xticks(rotation=25); plt.title("3: sampled action gets reinforced"); plt.ylabel("gradient contribution"); plt.show()

▶ What you'll see: a positive advantage increases the sampled action's logit and decreases competing logits.

*Why it's done this way:* we cannot backpropagate through the discrete random draw itself, but we can differentiate the log-probability of the action that was drawn. Multiplying by return gives an unbiased Monte Carlo estimate of the policy gradient: actions that led to above-baseline outcomes are made more probable in proportion to how surprising and how useful they were.

### 4. REINFORCE: update the policy from an episode

The vanilla REINFORCE estimator sums $\nabla_\theta\log\pi_\theta(a_t\mid s_t)G_t$ over an episode. In a tabular softmax policy, each state owns a row of logits, and only the row for the visited state receives a gradient at that time step.

In [ ]:
Theta_w = np.array([[0.0, 0.0], [0.0, 0.0]])  # two states, two actions: logits are all tied.
states_w = np.array([0, 1])  # one sampled two-step episode visited state 0 then state 1.
actions_w = np.array([1, 0])  # sampled right in state 0, then left in state 1.
returns_ep_w = np.array([2.62, 1.8])  # discounted returns attached to those decisions.
print("Theta shape:", Theta_w.shape)
print("episode states/actions:", list(zip(states_w, actions_w)))

▶ What you'll see: the policy table has shape `|S| × |A|`, and each episode step points to one row.

In [ ]:
def softmax_w(x_w):
    z_w = x_w - np.max(x_w)
    e_w = np.exp(z_w)
    return e_w / np.sum(e_w)

grad_w = np.zeros_like(Theta_w)  # accumulate the episode gradient.
for s_w, a_w, G_w in zip(states_w, actions_w, returns_ep_w):
    p_s_w = softmax_w(Theta_w[s_w])  # current action probabilities in this state.
    grad_log_w = -p_s_w  # start with -pi for every action.
    grad_log_w[a_w] += 1.0  # add 1 for the sampled action: one_hot(a)-pi.
    grad_w[s_w] += G_w * grad_log_w  # weight by return.
print("episode gradient:\n", np.round(grad_w, 3))

▶ What you'll see: state 0 reinforces action 1, while state 1 reinforces action 0.

In [ ]:
eta_w = 0.2  # learning rate for gradient ascent on expected return.
Theta_new_w = Theta_w + eta_w * grad_w  # ascent, not descent: we want larger J(theta).
print("updated logits:\n", np.round(Theta_new_w, 3))
print("new pi(s0):", np.round(softmax_w(Theta_new_w[0]), 3))
print("new pi(s1):", np.round(softmax_w(Theta_new_w[1]), 3))

▶ What you'll see: the sampled rewarded actions now have probabilities above 0.5 in their states.

*Why it's done this way:* a policy-gradient update is stochastic gradient **ascent** because the objective is expected return, not loss. The return $G_t$ acts like a credit-assignment weight: the same sampled action receives a larger log-probability push when the later consequence was larger.

### 5. Baselines and advantage reduce variance without changing the target

Raw returns can be noisy. Subtracting a baseline $b(s_t)$ gives an advantage $A_t=G_t-b(s_t)$, so actions are reinforced only when they did better than expected for that state. A baseline that does not depend on the sampled action keeps the policy gradient unbiased but can greatly reduce variance.

In [ ]:
sample_returns_w = np.array([0.0, 2.0, 4.0, 2.0, 6.0])  # noisy returns from the same state.
baseline_w = float(np.mean(sample_returns_w))  # a simple state-value estimate.
advantages_w = sample_returns_w - baseline_w  # centered consequences.
print("baseline:", round(baseline_w, 3))
print("advantages:", np.round(advantages_w, 3))
assert round(baseline_w, 3) == 2.800

▶ What you'll see: returns above 2.8 are positive advantages; returns below 2.8 become negative evidence.

In [ ]:
grad_component_w = np.array([0.25, -0.25])  # pretend the same action's log-prob gradient repeats.
raw_estimates_w = sample_returns_w[:, None] * grad_component_w  # no baseline.
centered_estimates_w = advantages_w[:, None] * grad_component_w  # with baseline.
print("raw variance first coordinate:", round(float(np.var(raw_estimates_w[:, 0])), 3))
print("centered variance first coordinate:", round(float(np.var(centered_estimates_w[:, 0])), 3))

▶ What you'll see: this constant baseline centers the samples; in real episodes it usually lowers gradient noise.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(np.arange(len(sample_returns_w)) - 0.18, sample_returns_w, width=0.36, label="G")
plt.bar(np.arange(len(sample_returns_w)) + 0.18, advantages_w, width=0.36, label="A=G-b")
plt.axhline(0, color="black", linewidth=0.8); plt.axhline(baseline_w, color="gray", linestyle="--", label="baseline")
plt.title("5: baseline turns returns into advantages"); plt.legend(); plt.show()

▶ What you'll see: advantages are centered around zero, so mediocre outcomes stop receiving automatic positive reinforcement.

*Why it's done this way:* adding or subtracting an action-independent baseline has expectation zero inside $\mathbb{E}[\nabla\log\pi(a\mid s)b(s)]$ because score-function gradients sum to zero over actions. The learning target remains expected return, but the samples become easier to average because the update says "better or worse than usual" rather than "positive reward means push everything up."

### 6. Training a toy policy from scratch

Now we train a tiny one-state environment. Action 0 gives reward 0; action 1 gives reward 1. The optimal policy should put most probability on action 1. This is not a realistic environment, but it isolates the REINFORCE mechanics: sample an action, observe a return, form $\nabla\log\pi(a)$, optionally subtract a baseline, and ascend.

In [ ]:
rng_w = np.random.default_rng(0)  # local generator for reproducible training.
theta_w = np.array([0.0, 0.0])  # two action logits for the one-state policy.
baseline_train_w = 0.0  # running average return baseline.
alpha_w = 0.12  # policy learning rate.
beta_w = 0.05  # baseline learning rate.
print("initial policy:", np.round(softmax_w(theta_w), 3))

▶ What you'll see: the policy starts uniform, so each action has probability 0.5.

In [ ]:
p1_curve_w, reward_curve_w = [], []
for episode_w in range(250):
    probs_ep_w = softmax_w(theta_w)  # current policy.
    action_w = int(rng_w.choice(2, p=probs_ep_w))  # sample from the policy, not argmax.
    reward_w = 1.0 if action_w == 1 else 0.0  # toy environment reward.
    advantage_ep_w = reward_w - baseline_train_w  # baseline-centered return.
    grad_log_ep_w = -probs_ep_w
    grad_log_ep_w[action_w] += 1.0  # one_hot(action)-pi.
    theta_w += alpha_w * advantage_ep_w * grad_log_ep_w  # REINFORCE ascent step.
    baseline_train_w += beta_w * (reward_w - baseline_train_w)  # update baseline estimate.
    p1_curve_w.append(softmax_w(theta_w)[1])
    reward_curve_w.append(reward_w)
print("final policy:", np.round(softmax_w(theta_w), 3))
assert softmax_w(theta_w)[1] > 0.85

▶ What you'll see: action 1 becomes much more likely than action 0 after repeated sampled updates.

In [ ]:
window_w = 20
smooth_reward_w = np.convolve(reward_curve_w, np.ones(window_w) / window_w, mode="valid")
plt.figure(figsize=(5.2, 3.2))
plt.plot(p1_curve_w, label="P(action 1)", color="teal")
plt.plot(np.arange(window_w - 1, len(reward_curve_w)), smooth_reward_w, label="20-episode avg reward", color="darkorange")
plt.ylim(0, 1.05); plt.xlabel("episode"); plt.title("6: REINFORCE learning curve")
plt.legend(); plt.show()

▶ What you'll see: both the probability of the rewarding action and the smoothed reward trend upward; the curve is noisy because actions are sampled.

*Why it's done this way:* sampling is essential because the gradient estimator is defined under the current policy distribution. The baseline makes a zero reward below expectation and a one reward above expectation, so the same code can both discourage bad sampled actions and reinforce good sampled actions without ever needing supervised labels.

## 🛠️ Setup

In [ ]:
import numpy as np  # load NumPy for arrays, softmax probabilities, random sampling, and numerical checks.
import matplotlib.pyplot as plt  # load Matplotlib for policy, return, and learning-curve visualizations.
np.random.seed(0)  # make global-random examples reproducible.

def softmax(logits):  # convert unconstrained logits into a valid probability vector.
    logits = np.asarray(logits, dtype=float)  # ensure vector arithmetic is predictable.
    shifted = logits - np.max(logits)  # subtract max for numerical stability without changing probabilities.
    weights = np.exp(shifted)  # exponentiate to get positive unnormalized probabilities.
    return weights / np.sum(weights)  # normalize so probabilities sum to 1.

def discounted_returns(rewards, gamma):  # compute every G_t for one completed episode.
    rewards = np.asarray(rewards, dtype=float)  # accept lists while doing float arithmetic.
    out = np.zeros_like(rewards, dtype=float)  # allocate return vector.
    running = 0.0  # stores G_{t+1} while moving backward.
    for t in range(len(rewards) - 1, -1, -1):  # walk from final reward back to the first.
        running = rewards[t] + gamma * running  # Bellman-style return recurrence.
        out[t] = running  # save G_t.
    return out  # return one Monte Carlo target per time step.

def grad_log_softmax(probs, action):  # gradient of log pi(action) with respect to logits.
    grad = -np.asarray(probs, dtype=float).copy()  # derivative starts as -pi for every action.
    grad[int(action)] += 1.0  # chosen action receives the +1 term from one_hot(action).
    return grad  # equals one_hot(action) - pi.

def sample_action(logits, rng):  # draw one action from a softmax policy.
    probs = softmax(logits)  # turn logits into probabilities.
    action = int(rng.choice(len(probs), p=probs))  # sample according to the current policy.
    return action, probs  # return both for learning diagnostics.

def moving_average(x, width):  # smooth a noisy curve for plots.
    x = np.asarray(x, dtype=float)  # convert to array.
    return np.convolve(x, np.ones(width) / width, mode="valid")  # centered-looking running mean without extra packages.

## 🟢 Basics (warm-up)

### Basic 1 — Compute one discounted return

**Goal.** Turn a short reward stream into one return, because REINFORCE weights actions by delayed consequence rather than immediate reward alone. We build it in 2 steps.

In [ ]:
rewards_b1 = np.array([1.0, 0.0, 2.0])  # define a three-step reward stream.
gamma_b1 = 0.9  # set the discount from the lesson text.
terms_b1 = rewards_b1 * (gamma_b1 ** np.arange(len(rewards_b1)))  # compute each discounted contribution.
print("discounted terms:", np.round(terms_b1, 3))  # inspect the pieces of G0.

▶ What you'll see: reward 2 contributes 1.620 because it arrives two steps later.

In [ ]:
G_b1 = float(np.sum(terms_b1))  # sum discounted rewards into the return.
print("G0:", round(G_b1, 3))  # inspect the final return.
assert round(G_b1, 3) == 2.620  # verify the concrete lesson number.
plt.figure(figsize=(4, 3))
plt.bar(["r0", "γr1", "γ²r2"], terms_b1, color="teal")
plt.title("Basic 1: discounted-return terms"); plt.ylabel("contribution"); plt.show()

▶ What you'll see: the return is larger than the immediate reward because later reward still counts.

👀 Takeaway: REINFORCE assigns credit using return $G$, not just the first reward seen after an action.

### Basic 2 — Compute all returns in an episode

**Goal.** Compute $G_t$ for every time step, because each sampled action receives the return from that point onward. We build it in 2 steps.

In [ ]:
rewards_b2 = np.array([0.0, 1.0, 0.0, 3.0])  # define one completed episode.
gamma_b2 = 0.8  # use a smaller discount for easy inspection.
returns_b2 = discounted_returns(rewards_b2, gamma_b2)  # compute all Monte Carlo targets.
print("rewards:", rewards_b2)  # inspect raw rewards.
print("returns:", np.round(returns_b2, 3))  # inspect G_t values.

▶ What you'll see: earlier steps include more future reward, so their returns can be larger.

In [ ]:
assert round(float(returns_b2[0]), 3) == 2.336  # 0 + .8*1 + .8^3*3.
plt.figure(figsize=(4.5, 3))
plt.plot(returns_b2, marker="o", color="purple")
plt.title("Basic 2: G_t for each step"); plt.xlabel("time t"); plt.ylabel("return"); plt.show()

▶ What you'll see: the curve shows how future reward propagates backward through the episode.

👀 Takeaway: a single episode supplies many policy-gradient targets, one per visited state-action pair.

### Basic 3 — Convert logits to probabilities

**Goal.** Use softmax on two logits, because a policy must choose actions with probabilities that sum to one. We build it in 2 steps.

In [ ]:
logits_b3 = np.array([1.0, 0.0])  # define two action preferences.
probs_b3 = softmax(logits_b3)  # normalize them into a policy distribution.
print("probabilities:", np.round(probs_b3, 3))  # inspect pi(a|s).
assert np.allclose(np.round(probs_b3, 3), [0.731, 0.269])  # verify the lesson softmax.

▶ What you'll see: action 0 is more likely but action 1 still has probability mass.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["action 0", "action 1"], probs_b3, color="darkorange")
plt.ylim(0, 1); plt.title("Basic 3: softmax policy"); plt.ylabel("probability"); plt.show()

▶ What you'll see: the larger logit receives the larger probability.

👀 Takeaway: policy gradients optimize logits, while softmax enforces valid stochastic action probabilities.

### Basic 4 — Expected reward under a policy

**Goal.** Average action rewards by policy probabilities, because the objective is expected consequence under the current policy. We build it in 2 steps.

In [ ]:
probs_b4 = softmax(np.array([1.0, 0.0]))  # reproduce the lesson policy.
action_rewards_b4 = np.array([2.0, 0.0])  # rewards if each action were chosen.
weighted_b4 = probs_b4 * action_rewards_b4  # probability-weighted rewards.
print("weighted rewards:", np.round(weighted_b4, 3))  # inspect each action's contribution.

▶ What you'll see: only action 0 contributes because action 1 has reward zero.

In [ ]:
expected_b4 = float(np.sum(weighted_b4))  # compute expected reward.
print("expected reward:", round(expected_b4, 3))  # inspect the scalar objective value.
assert round(expected_b4, 3) == 1.462  # verify the concrete lesson number.
plt.figure(figsize=(4, 3))
plt.bar(["a0", "a1", "E[R]"], [weighted_b4[0], weighted_b4[1], expected_b4], color="seagreen")
plt.title("Basic 4: probability-weighted reward"); plt.show()

▶ What you'll see: expected reward is the sum of probability-weighted action outcomes.

👀 Takeaway: improving a policy means moving probability mass toward actions with better expected return.

### Basic 5 — Sample an action from a policy

**Goal.** Draw actions stochastically, because REINFORCE learns from actions sampled by the current policy, not from a hard argmax. We build it in 2 steps.

In [ ]:
rng_b5 = np.random.default_rng(5)  # create a reproducible random generator.
logits_b5 = np.array([0.2, 1.0])  # action 1 is preferred but action 0 remains possible.
actions_b5 = np.array([sample_action(logits_b5, rng_b5)[0] for _ in range(12)])  # sample a few actions.
print("sampled actions:", actions_b5)  # inspect stochastic choices.

▶ What you'll see: action 1 appears more often, but action 0 can still be sampled.

In [ ]:
counts_b5 = np.bincount(actions_b5, minlength=2)  # count sampled actions.
print("counts:", counts_b5, "policy:", np.round(softmax(logits_b5), 3))  # compare samples to probabilities.
plt.figure(figsize=(4, 3))
plt.bar(["a0", "a1"], counts_b5, color="slateblue")
plt.title("Basic 5: sampled action counts"); plt.ylabel("count in 12 draws"); plt.show()

▶ What you'll see: the sample histogram roughly follows the policy but has random noise.

👀 Takeaway: stochastic sampling creates exploration and also creates variance in the gradient estimate.

### Basic 6 — Gradient of log softmax

**Goal.** Compute $\nabla_z\log\pi(a)$ for logits, because this is the score-function term in REINFORCE. We build it in 2 steps.

In [ ]:
probs_b6 = softmax(np.array([1.0, 0.0]))  # compute pi for two actions.
action_b6 = 0  # suppose action 0 was sampled.
grad_b6 = grad_log_softmax(probs_b6, action_b6)  # one_hot(action)-pi.
print("grad log pi(a0):", np.round(grad_b6, 3))  # inspect the two logit derivatives.
assert np.allclose(np.round(grad_b6, 3), [0.269, -0.269])

▶ What you'll see: the chosen action's logit gets a positive derivative and the other gets a negative derivative.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["logit 0", "logit 1"], grad_b6, color=["green", "red"])
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Basic 6: log-policy gradient"); plt.ylabel("derivative"); plt.show()

▶ What you'll see: the derivatives sum to zero because softmax moves probability mass between actions.

👀 Takeaway: log-policy gradients say how to change logits to make the sampled action more likely.

### Basic 7 — Weight a log-probability gradient by return

**Goal.** Multiply the score-function gradient by return, because REINFORCE reinforces actions in proportion to consequence. We build it in 2 steps.

In [ ]:
probs_b7 = softmax(np.array([1.0, 0.0]))  # current two-action policy.
grad_b7 = grad_log_softmax(probs_b7, 0)  # gradient for sampled action 0.
G_b7 = 2.62  # return from the worked discounted-reward example.
reinforce_dir_b7 = G_b7 * grad_b7  # policy-gradient sample contribution.
print("REINFORCE direction:", np.round(reinforce_dir_b7, 3))  # inspect weighted update.

▶ What you'll see: a positive return scales up the push toward the sampled action.

In [ ]:
assert np.allclose(np.round(reinforce_dir_b7, 3), [0.705, -0.705])  # 2.62 * [0.269, -0.269].
plt.figure(figsize=(4, 3))
plt.bar(["logit 0", "logit 1"], reinforce_dir_b7, color=["green", "red"])
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Basic 7: return-weighted gradient"); plt.show()

▶ What you'll see: the same gradient shape becomes larger when the return is larger.

👀 Takeaway: REINFORCE is log-probability gradient times a return or advantage weight.

### Basic 8 — Subtract a baseline to get advantage

**Goal.** Convert returns into advantages, because actions should be judged relative to what was expected in that state. We build it in 2 steps.

In [ ]:
returns_b8 = np.array([0.0, 2.0, 4.0, 2.0, 6.0])  # example returns from repeated visits.
baseline_b8 = float(np.mean(returns_b8))  # simple value baseline.
adv_b8 = returns_b8 - baseline_b8  # centered advantages.
print("baseline:", round(baseline_b8, 3))  # inspect the expected return.
print("advantages:", np.round(adv_b8, 3))  # inspect better/worse-than-usual values.
assert round(baseline_b8, 3) == 2.800

▶ What you'll see: returns below 2.8 become negative advantages even though rewards may be nonnegative.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.bar(range(len(returns_b8)), adv_b8, color="royalblue")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Basic 8: advantage = return - baseline"); plt.ylabel("advantage"); plt.show()

▶ What you'll see: advantage bars center around zero, separating good and bad outcomes relative to expectation.

👀 Takeaway: baselines reduce variance by making the update about surprise, not raw reward scale.

### Basic 9 — Update logits once

**Goal.** Perform one REINFORCE ascent step, because policy gradients directly change policy parameters. We build it in 3 steps.

In [ ]:
theta_b9 = np.array([0.0, 0.0])  # start with a uniform two-action policy.
probs_b9 = softmax(theta_b9)  # current action probabilities.
action_b9 = 1  # suppose action 1 was sampled.
adv_b9 = 1.0  # suppose it did better than baseline.
print("before probs:", np.round(probs_b9, 3))  # inspect current policy.

▶ What you'll see: the initial policy is `[0.5, 0.5]`.

In [ ]:
eta_b9 = 0.4  # learning rate for this one update.
grad_b9 = grad_log_softmax(probs_b9, action_b9)  # log-policy gradient for sampled action.
theta_new_b9 = theta_b9 + eta_b9 * adv_b9 * grad_b9  # ascent update.
print("updated logits:", np.round(theta_new_b9, 3))  # inspect parameter movement.

In [ ]:
probs_new_b9 = softmax(theta_new_b9)  # convert updated logits to probabilities.
print("after probs:", np.round(probs_new_b9, 3))  # inspect the policy shift.
assert probs_new_b9[1] > 0.5  # sampled positive-advantage action became more likely.
plt.figure(figsize=(4, 3))
plt.bar(["before a1", "after a1"], [probs_b9[1], probs_new_b9[1]], color="seagreen")
plt.ylim(0, 1); plt.title("Basic 9: one policy update"); plt.ylabel("P(action 1)"); plt.show()

▶ What you'll see: the probability of action 1 rises above 0.5 after one positive-advantage update.

👀 Takeaway: policy-gradient ascent changes logits so successful sampled actions become more likely.

### Basic 10 — Plot a tiny learning curve

**Goal.** Train on a one-state bandit long enough to see learning, because REINFORCE curves are noisy but should trend upward on a simple problem. We build it in 3 steps.

In [ ]:
rng_b10 = np.random.default_rng(10)  # reproducible sampling.
theta_b10 = np.array([0.0, 0.0])  # two action logits.
base_b10 = 0.0  # running reward baseline.
p1_b10, rewards_b10 = [], []  # store diagnostics.
print("initial P(a1):", round(float(softmax(theta_b10)[1]), 3))

▶ What you'll see: the rewarding action starts at probability 0.5.

In [ ]:
for ep_b10 in range(160):  # train for a small number of episodes.
    action_b10, probs_b10 = sample_action(theta_b10, rng_b10)  # sample from current policy.
    reward_b10 = 1.0 if action_b10 == 1 else 0.0  # action 1 is the rewarding arm.
    adv_b10 = reward_b10 - base_b10  # baseline-centered return.
    theta_b10 += 0.15 * adv_b10 * grad_log_softmax(probs_b10, action_b10)  # REINFORCE update.
    base_b10 += 0.05 * (reward_b10 - base_b10)  # update running baseline.
    p1_b10.append(softmax(theta_b10)[1]); rewards_b10.append(reward_b10)  # save diagnostics.
print("final P(a1):", round(float(p1_b10[-1]), 3))
assert p1_b10[-1] > 0.75

In [ ]:
smooth_b10 = moving_average(rewards_b10, 20)  # smooth noisy sampled rewards.
plt.figure(figsize=(5, 3))
plt.plot(p1_b10, label="P(a1)", color="teal")
plt.plot(np.arange(19, len(rewards_b10)), smooth_b10, label="20-episode reward", color="orange")
plt.ylim(0, 1.05); plt.title("Basic 10: tiny REINFORCE curve"); plt.xlabel("episode"); plt.legend(); plt.show()

▶ What you'll see: the probability of the rewarding action climbs, and average reward follows with noise.

👀 Takeaway: even a simple REINFORCE learner produces noisy but improving learning curves.

## 🟡 Easy

### Easy 1 — REINFORCE on a two-action bandit

**Goal.** Implement a complete from-scratch REINFORCE loop for a bandit, because the bandit removes state transitions and exposes the policy-gradient core. We build it in 4 steps.

In [ ]:
rng_e1 = np.random.default_rng(11)  # reproducible bandit sampling.
theta_e1 = np.array([0.0, 0.0])  # policy logits for two actions.
reward_means_e1 = np.array([0.2, 1.0])  # action 1 is better in expectation.
probs_start_e1 = softmax(theta_e1)  # initial policy.
print("start policy:", np.round(probs_start_e1, 3))

▶ What you'll see: both actions start equally likely.

In [ ]:
p1_e1, rewards_e1 = [], []  # store diagnostics.
for ep_e1 in range(300):
    action_e1, probs_e1 = sample_action(theta_e1, rng_e1)  # sample action.
    reward_e1 = reward_means_e1[action_e1] + 0.1 * rng_e1.normal()  # noisy reward.
    theta_e1 += 0.08 * reward_e1 * grad_log_softmax(probs_e1, action_e1)  # vanilla REINFORCE.
    p1_e1.append(softmax(theta_e1)[1]); rewards_e1.append(reward_e1)  # save learning curve.
print("end policy:", np.round(softmax(theta_e1), 3))
assert softmax(theta_e1)[1] > 0.8

In [ ]:
avg_reward_e1 = moving_average(rewards_e1, 25)  # smooth reward noise.
print("last smoothed reward:", round(float(avg_reward_e1[-1]), 3))

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(p1_e1, label="P(best action)", color="teal")
plt.plot(np.arange(24, len(rewards_e1)), avg_reward_e1, label="25-episode avg reward", color="orange")
plt.title("Easy 1: bandit REINFORCE learning"); plt.xlabel("episode"); plt.legend(); plt.show()

▶ What you'll see: probability shifts toward action 1, and reward improves as that action is sampled more often.

👀 Takeaway: REINFORCE can optimize a stochastic policy without a value table when returns are sampled from the policy.

### Easy 2 — Train with a running baseline

**Goal.** Compare raw-return and baseline-centered updates on the same noisy bandit, because baselines usually make policy-gradient learning smoother. We build it in 4 steps.

In [ ]:
rng_e2_raw = np.random.default_rng(12)  # random stream for raw-return learner.
rng_e2_base = np.random.default_rng(12)  # matching stream for baseline learner.
theta_raw_e2 = np.array([0.0, 0.0])  # raw learner logits.
theta_base_e2 = np.array([0.0, 0.0])  # baseline learner logits.
baseline_e2 = 0.0  # running estimate of expected reward.
print("initialized two learners")

▶ What you'll see: both learners start from the same uniform policy.

In [ ]:
p_raw_e2, p_base_e2 = [], []  # store best-action probabilities.
for ep_e2 in range(350):
    a_raw_e2, pr_raw_e2 = sample_action(theta_raw_e2, rng_e2_raw)
    r_raw_e2 = (1.0 if a_raw_e2 == 1 else 0.1) + 0.4 * rng_e2_raw.normal()
    theta_raw_e2 += 0.05 * r_raw_e2 * grad_log_softmax(pr_raw_e2, a_raw_e2)
    a_base_e2, pr_base_e2 = sample_action(theta_base_e2, rng_e2_base)
    r_base_e2 = (1.0 if a_base_e2 == 1 else 0.1) + 0.4 * rng_e2_base.normal()
    adv_base_e2 = r_base_e2 - baseline_e2
    theta_base_e2 += 0.05 * adv_base_e2 * grad_log_softmax(pr_base_e2, a_base_e2)
    baseline_e2 += 0.05 * (r_base_e2 - baseline_e2)
    p_raw_e2.append(softmax(theta_raw_e2)[1]); p_base_e2.append(softmax(theta_base_e2)[1])
print("final P(best), raw/base:", round(p_raw_e2[-1], 3), round(p_base_e2[-1], 3))

In [ ]:
assert p_base_e2[-1] > 0.75  # baseline learner should still learn the better action.
print("final baseline estimate:", round(baseline_e2, 3))

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(p_raw_e2, label="raw return", alpha=0.8)
plt.plot(p_base_e2, label="with baseline", alpha=0.8)
plt.ylim(0, 1.05); plt.title("Easy 2: baseline-centered learning"); plt.xlabel("episode"); plt.legend(); plt.show()

▶ What you'll see: both policies improve, while the baseline version uses relative outcomes rather than always-positive rewards.

👀 Takeaway: a baseline changes update variance and direction on mediocre samples without changing the expected policy-gradient target.

### Easy 3 — Two-state tabular softmax policy

**Goal.** Update only the visited state's logit row, because a tabular policy has separate action probabilities for each state. We build it in 4 steps.

In [ ]:
Theta_e3 = np.zeros((2, 2))  # two states by two actions.
states_e3 = np.array([0, 1, 0])  # states visited in one sampled episode.
actions_e3 = np.array([1, 0, 1])  # actions sampled in those states.
rewards_e3 = np.array([0.0, 1.0, 2.0])  # reward stream.
returns_e3 = discounted_returns(rewards_e3, 0.9)  # Monte Carlo targets.
print("returns:", np.round(returns_e3, 3))

▶ What you'll see: the first action receives credit for later rewards.

In [ ]:
grad_e3 = np.zeros_like(Theta_e3)  # accumulate one episode gradient.
for s_e3, a_e3, G_e3 in zip(states_e3, actions_e3, returns_e3):
    probs_e3 = softmax(Theta_e3[s_e3])
    grad_e3[s_e3] += G_e3 * grad_log_softmax(probs_e3, a_e3)
print("gradient table:\n", np.round(grad_e3, 3))

In [ ]:
Theta_new_e3 = Theta_e3 + 0.2 * grad_e3  # gradient ascent update.
print("new policy state 0:", np.round(softmax(Theta_new_e3[0]), 3))
print("new policy state 1:", np.round(softmax(Theta_new_e3[1]), 3))
assert softmax(Theta_new_e3[0])[1] > 0.5 and softmax(Theta_new_e3[1])[0] > 0.5

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.imshow(np.vstack([softmax(Theta_new_e3[0]), softmax(Theta_new_e3[1])]), cmap="viridis", vmin=0, vmax=1)
plt.colorbar(label="probability"); plt.title("Easy 3: updated tabular policy"); plt.xlabel("action"); plt.ylabel("state"); plt.show()

▶ What you'll see: each state's sampled high-return action becomes more probable in that state only.

👀 Takeaway: policy parameters are local to the state features or table rows that produced the sampled action.

### Easy 4 — Estimate gradient variance

**Goal.** Measure how sampled gradient estimates vary with and without a baseline, because high variance is the main practical weakness of vanilla REINFORCE. We build it in 4 steps.

In [ ]:
rng_e4 = np.random.default_rng(14)  # reproducible Monte Carlo experiment.
probs_e4 = softmax(np.array([0.0, 0.0]))  # uniform two-action policy.
returns_by_action_e4 = np.array([0.2, 1.0])  # action 1 has higher mean return.
baseline_e4 = float(np.dot(probs_e4, returns_by_action_e4))  # exact expected reward baseline.
print("baseline:", round(baseline_e4, 3))

▶ What you'll see: the baseline is the policy's current expected reward.

In [ ]:
raw_grads_e4, centered_grads_e4 = [], []  # store first logit gradient estimates.
for _ in range(500):
    action_e4 = int(rng_e4.choice(2, p=probs_e4))
    G_e4 = returns_by_action_e4[action_e4] + 0.5 * rng_e4.normal()
    score_e4 = grad_log_softmax(probs_e4, action_e4)[0]
    raw_grads_e4.append(G_e4 * score_e4)
    centered_grads_e4.append((G_e4 - baseline_e4) * score_e4)
raw_grads_e4 = np.array(raw_grads_e4); centered_grads_e4 = np.array(centered_grads_e4)
print("variance raw/centered:", round(float(np.var(raw_grads_e4)), 4), round(float(np.var(centered_grads_e4)), 4))

In [ ]:
assert np.var(centered_grads_e4) < np.var(raw_grads_e4)  # baseline lowers variance in this setup.
print("mean raw/centered:", round(float(np.mean(raw_grads_e4)), 4), round(float(np.mean(centered_grads_e4)), 4))

In [ ]:
plt.figure(figsize=(5, 3))
plt.hist(raw_grads_e4, bins=30, alpha=0.6, label="raw")
plt.hist(centered_grads_e4, bins=30, alpha=0.6, label="baseline")
plt.title("Easy 4: gradient-estimate spread"); plt.legend(); plt.show()

▶ What you'll see: the centered histogram is narrower while keeping a similar average gradient direction.

👀 Takeaway: baselines are variance-reduction tools, not a way to change which policy is optimal.

### Easy 5 — Normalize returns within a batch

**Goal.** Standardize a batch of returns, because small teaching implementations often normalize advantages to make step sizes easier to tune. We build it in 4 steps.

In [ ]:
returns_e5 = np.array([1.0, 3.0, 2.0, 6.0, 4.0])  # batch of Monte Carlo returns.
mean_e5 = float(np.mean(returns_e5))  # batch baseline.
std_e5 = float(np.std(returns_e5) + 1e-8)  # scale with epsilon for safety.
adv_e5 = (returns_e5 - mean_e5) / std_e5  # normalized advantages.
print("mean/std:", round(mean_e5, 3), round(std_e5, 3))

▶ What you'll see: the batch has mean 3.2 and a nonzero spread.

In [ ]:
print("normalized advantages:", np.round(adv_e5, 3))
assert abs(float(np.mean(adv_e5))) < 1e-10  # standardized advantages center to zero.
assert round(float(np.std(adv_e5)), 3) == 1.000  # and have unit standard deviation.

In [ ]:
score_e5 = np.array([0.5, -0.5])  # pretend all samples used the same chosen-action score.
updates_e5 = adv_e5[:, None] * score_e5  # scaled update contributions.
print("first three updates:\n", np.round(updates_e5[:3], 3))

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.bar(range(len(adv_e5)), adv_e5, color="mediumpurple")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Easy 5: normalized advantages"); plt.ylabel("z-scored advantage"); plt.show()

▶ What you'll see: positive and negative updates are on a controlled, roughly unit scale.

👀 Takeaway: advantage normalization is a practical scale trick; the conceptual baseline is still expected return.

## 🔴 Advanced

### Advanced 1 — Train a delayed-reward two-step policy

**Goal.** Learn from a reward that arrives only after two decisions, because delayed credit assignment is why returns matter. We build it in 5 steps.

In [ ]:
rng_a1 = np.random.default_rng(21)  # reproducible episode sampling.
Theta_a1 = np.zeros((2, 2))  # state 0 and state 1, each with two action logits.
baseline_a1 = np.zeros(2)  # running value baseline per state.
prob_good_a1, episode_return_a1 = [], []  # diagnostics.
print("initial policies:", np.round([softmax(Theta_a1[0]), softmax(Theta_a1[1])], 3))

▶ What you'll see: both states begin with uniform action probabilities.

In [ ]:
for ep_a1 in range(450):
    states_a1 = [0, 1]
    a0_a1, p0_a1 = sample_action(Theta_a1[0], rng_a1)
    a1_a1, p1_a1 = sample_action(Theta_a1[1], rng_a1)
    rewards_a1 = np.array([0.0, 1.0 if (a0_a1 == 1 and a1_a1 == 1) else 0.0])
    returns_a1 = discounted_returns(rewards_a1, 0.95)
    for s_a1, a_a1, probs_a1, G_a1 in [(0, a0_a1, p0_a1, returns_a1[0]), (1, a1_a1, p1_a1, returns_a1[1])]:
        adv_a1 = G_a1 - baseline_a1[s_a1]
        Theta_a1[s_a1] += 0.18 * adv_a1 * grad_log_softmax(probs_a1, a_a1)
        baseline_a1[s_a1] += 0.04 * (G_a1 - baseline_a1[s_a1])
    prob_good_a1.append(softmax(Theta_a1[0])[1] * softmax(Theta_a1[1])[1])
    episode_return_a1.append(float(np.sum(rewards_a1)))
print("final policies:", np.round([softmax(Theta_a1[0]), softmax(Theta_a1[1])], 3))

In [ ]:
assert softmax(Theta_a1[0])[1] > 0.8 and softmax(Theta_a1[1])[1] > 0.8  # both delayed-credit actions learned.
print("final probability of successful action pair:", round(float(prob_good_a1[-1]), 3))

In [ ]:
smooth_a1 = moving_average(episode_return_a1, 30)  # smooth binary returns.
plt.figure(figsize=(5, 3))
plt.plot(prob_good_a1, label="P(a0=1)P(a1=1)", color="teal")
plt.plot(np.arange(29, len(episode_return_a1)), smooth_a1, label="30-episode success", color="orange")
plt.ylim(0, 1.05); plt.title("Advanced 1: delayed-reward REINFORCE"); plt.xlabel("episode"); plt.legend(); plt.show()

▶ What you'll see: both the probability of the successful two-action sequence and the smoothed success rate rise together.

In [ ]:
print("learned baselines:", np.round(baseline_a1, 3))  # inspect state-value estimates after training.

▶ What you'll see: state baselines approximate the returns usually seen from those states under the learned policy.

👀 Takeaway: Monte Carlo returns let an early action receive credit for a reward that appears only after later choices.

### Advanced 2 — Reward scaling changes step size

**Goal.** Show why return scale matters, because multiplying all rewards multiplies the policy-gradient update unless advantages are normalized or learning rates are retuned. We build it in 4 steps.

In [ ]:
probs_a2 = softmax(np.array([0.0, 0.0]))  # uniform policy.
action_a2 = 1  # sampled action.
score_a2 = grad_log_softmax(probs_a2, action_a2)  # log-policy gradient.
returns_a2 = np.array([1.0, 10.0, 100.0])  # same outcome pattern at different scales.
print("score:", score_a2)

▶ What you'll see: the score vector is fixed before reward scaling is applied.

In [ ]:
updates_a2 = np.array([G_a2 * score_a2 for G_a2 in returns_a2])  # raw REINFORCE directions.
norms_a2 = np.linalg.norm(updates_a2, axis=1)  # magnitude of each update.
print("update norms:", np.round(norms_a2, 3))
assert round(float(norms_a2[2] / norms_a2[0]), 1) == 100.0

In [ ]:
normalized_returns_a2 = (returns_a2 - np.mean(returns_a2)) / np.std(returns_a2)  # batch scaling.
updates_normed_a2 = np.array([G_a2 * score_a2 for G_a2 in normalized_returns_a2])
print("normalized returns:", np.round(normalized_returns_a2, 3))

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(["G=1", "G=10", "G=100"], norms_a2, color="crimson")
plt.yscale("log"); plt.title("Advanced 2: reward scale multiplies updates"); plt.ylabel("update norm, log scale"); plt.show()

▶ What you'll see: a 100× larger return creates a 100× larger raw gradient step.

👀 Takeaway: reward scaling is an optimization issue even when it does not change the ranking of policies.

### Advanced 3 — Entropy bonus keeps exploration alive

**Goal.** Add an entropy-gradient term, because policy gradients can become prematurely deterministic before enough exploration has happened. We build it in 5 steps.

In [ ]:
logits_a3 = np.array([2.0, -1.0])  # already confident policy.
probs_a3 = softmax(logits_a3)  # action probabilities.
entropy_a3 = -float(np.sum(probs_a3 * np.log(probs_a3 + 1e-12)))  # categorical entropy.
print("policy:", np.round(probs_a3, 3), "entropy:", round(entropy_a3, 3))

▶ What you'll see: the dominant action has high probability and entropy is low.

In [ ]:
# finite-difference entropy gradient keeps the code explicit and dependency-free for the teaching demo.
eps_a3 = 1e-5
grad_entropy_a3 = np.zeros_like(logits_a3)
for i_a3 in range(len(logits_a3)):
    plus_a3 = logits_a3.copy(); plus_a3[i_a3] += eps_a3
    minus_a3 = logits_a3.copy(); minus_a3[i_a3] -= eps_a3
    H_plus_a3 = -np.sum(softmax(plus_a3) * np.log(softmax(plus_a3) + 1e-12))
    H_minus_a3 = -np.sum(softmax(minus_a3) * np.log(softmax(minus_a3) + 1e-12))
    grad_entropy_a3[i_a3] = (H_plus_a3 - H_minus_a3) / (2 * eps_a3)
print("entropy gradient:", np.round(grad_entropy_a3, 3))

In [ ]:
action_a3 = 0  # sampled dominant action.
adv_a3 = 1.0  # good outcome reinforces it.
beta_a3 = 0.2  # entropy bonus weight.
plain_update_a3 = adv_a3 * grad_log_softmax(probs_a3, action_a3)
entropy_update_a3 = plain_update_a3 + beta_a3 * grad_entropy_a3
print("plain update:", np.round(plain_update_a3, 3))
print("entropy-regularized update:", np.round(entropy_update_a3, 3))

In [ ]:
new_plain_a3 = softmax(logits_a3 + 0.5 * plain_update_a3)
new_entropy_a3 = softmax(logits_a3 + 0.5 * entropy_update_a3)
print("P(minority action) plain/entropy:", round(float(new_plain_a3[1]), 3), round(float(new_entropy_a3[1]), 3))
assert new_entropy_a3[1] > new_plain_a3[1]

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.bar(["before", "plain", "+entropy"], [probs_a3[1], new_plain_a3[1], new_entropy_a3[1]], color=["gray", "red", "teal"])
plt.title("Advanced 3: entropy preserves support"); plt.ylabel("P(minority action)"); plt.show()

▶ What you'll see: the entropy bonus leaves more probability on the minority action than the plain reinforcing update.

👀 Takeaway: entropy regularization trades a bit of greediness for continued exploration and safer policy updates.

### Advanced 4 — Check the score-function estimator by Monte Carlo

**Goal.** Verify that REINFORCE estimates the true gradient of expected reward in a two-action bandit, because the log-derivative trick can feel magical until the averages match. We build it in 5 steps.

In [ ]:
theta_a4 = np.array([0.3, -0.2])  # policy logits.
probs_a4 = softmax(theta_a4)  # current probabilities.
rewards_a4 = np.array([0.0, 2.0])  # deterministic reward for each action.
true_grad_a4 = probs_a4 * (rewards_a4 - float(np.dot(probs_a4, rewards_a4)))  # exact softmax gradient wrt logits.
print("policy:", np.round(probs_a4, 3))
print("true gradient:", np.round(true_grad_a4, 3))

▶ What you'll see: the gradient decreases action 0's logit and increases action 1's logit.

In [ ]:
rng_a4 = np.random.default_rng(24)  # reproducible Monte Carlo samples.
estimates_a4 = []  # store REINFORCE gradient samples.
for _ in range(20000):
    action_a4 = int(rng_a4.choice(2, p=probs_a4))
    G_a4 = rewards_a4[action_a4]
    estimates_a4.append(G_a4 * grad_log_softmax(probs_a4, action_a4))
mc_grad_a4 = np.mean(np.array(estimates_a4), axis=0)  # average estimator.
print("Monte Carlo gradient:", np.round(mc_grad_a4, 3))

In [ ]:
assert np.allclose(mc_grad_a4, true_grad_a4, atol=0.02)  # sampled estimator matches exact gradient.
error_a4 = mc_grad_a4 - true_grad_a4
print("estimation error:", np.round(error_a4, 4))

In [ ]:
labels_a4 = ["logit0", "logit1"]
x_a4 = np.arange(2)
plt.figure(figsize=(4.8, 3))
plt.bar(x_a4 - 0.18, true_grad_a4, width=0.36, label="true")
plt.bar(x_a4 + 0.18, mc_grad_a4, width=0.36, label="REINFORCE MC")
plt.xticks(x_a4, labels_a4); plt.axhline(0, color="black", linewidth=0.8)
plt.title("Advanced 4: estimator matches true gradient"); plt.legend(); plt.show()

▶ What you'll see: the Monte Carlo bars nearly overlap the exact gradient bars.

In [ ]:
print("sample count:", len(estimates_a4))  # inspect why the Monte Carlo estimate is accurate.

▶ What you'll see: many samples average out the variance of individual stochastic updates.

👀 Takeaway: REINFORCE is noisy sample by sample, but its expectation is the correct policy gradient.

### Advanced 5 — Batch REINFORCE with validation curves

**Goal.** Train with mini-batches of episodes and track both return and policy probability, because batching reduces noise and produces clearer diagnostics. We build it in 5 steps.

In [ ]:
rng_a5 = np.random.default_rng(25)  # reproducible batch training.
theta_a5 = np.array([0.0, 0.0])  # one-state, two-action policy.
baseline_a5 = 0.0  # running baseline.
batch_size_a5 = 20  # episodes per policy update.
p_best_a5, batch_return_a5 = [], []  # diagnostics.
print("batch size:", batch_size_a5)

▶ What you'll see: training will aggregate 20 sampled gradients before each update.

In [ ]:
for batch_a5 in range(80):
    grads_a5 = []
    returns_batch_a5 = []
    for _ in range(batch_size_a5):
        action_a5, probs_a5 = sample_action(theta_a5, rng_a5)
        reward_a5 = (1.0 if action_a5 == 1 else 0.0) + 0.2 * rng_a5.normal()
        advantage_a5 = reward_a5 - baseline_a5
        grads_a5.append(advantage_a5 * grad_log_softmax(probs_a5, action_a5))
        returns_batch_a5.append(reward_a5)
    theta_a5 += 0.12 * np.mean(grads_a5, axis=0)  # one averaged policy-gradient step.
    baseline_a5 += 0.08 * (float(np.mean(returns_batch_a5)) - baseline_a5)  # slower baseline update.
    p_best_a5.append(softmax(theta_a5)[1])
    batch_return_a5.append(float(np.mean(returns_batch_a5)))
print("final policy:", np.round(softmax(theta_a5), 3))

In [ ]:
assert p_best_a5[-1] > 0.75  # the batched learner should prefer the rewarding action.
print("final mean return:", round(float(batch_return_a5[-1]), 3))

In [ ]:
smooth_return_a5 = moving_average(batch_return_a5, 5)  # smooth batch averages further.
print("best-action probability start/end:", round(float(p_best_a5[0]), 3), round(float(p_best_a5[-1]), 3))

In [ ]:
plt.figure(figsize=(5.2, 3.2))
plt.plot(p_best_a5, label="P(best action)", color="teal")
plt.plot(np.arange(4, len(batch_return_a5)), smooth_return_a5, label="5-batch avg return", color="orange")
plt.ylim(0, 1.05); plt.title("Advanced 5: batched REINFORCE diagnostics"); plt.xlabel("batch update"); plt.legend(); plt.show()

▶ What you'll see: batched updates produce a smoother climb in best-action probability and average return than single-episode updates.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["initial P(best)", "final P(best)"], [0.5, p_best_a5[-1]], color=["gray", "seagreen"])
plt.ylim(0, 1); plt.title("Advanced 5: policy improvement summary"); plt.show()

▶ What you'll see: the final policy puts substantially more probability on the rewarding action than the initial uniform policy.

👀 Takeaway: batching averages noisy score-function estimates, making REINFORCE easier to monitor while preserving the same gradient idea.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Policy gradients optimize the policy parameters that choose actions.

REINFORCE directly changes action probabilities using complete discounted returns. The lesson formula links probability, optimization, and advantage-weighted log-probability updates. Save a copy to Drive to edit.

In [ ]:

import math
import random
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np

SEED = 1113
rng = np.random.default_rng(SEED)
random.seed(SEED)

ACTIONS = np.array([
    [-1, 0],
    [1, 0],
    [0, -1],
    [0, 1],
])
ACTION_NAMES = np.array(["U", "D", "L", "R"])

@dataclass
class LadderEnv:
    name: str
    height: int
    width: int
    start: tuple
    goal: tuple
    walls: tuple
    slip: float
    wind: float
    step_cost: float
    goal_reward: float
    traps: dict
    max_steps: int
    continuous: bool = False

    @property
    def n_states(self):
        return self.height * self.width

    @property
    def n_actions(self):
        return len(ACTIONS)

    def state_index(self, pos):
        row, col = pos
        return row * self.width + col

    def index_state(self, idx):
        row = idx // self.width
        col = idx % self.width
        return (row, col)

    def reset(self):
        return self.state_index(self.start)

    def move(self, pos, action, local_rng):
        actual = int(action)
        if local_rng.random() < self.slip:
            actual = int(local_rng.integers(0, self.n_actions))
        delta = ACTIONS[actual].copy()
        if self.wind > 0.0 and local_rng.random() < self.wind:
            delta = delta + np.array([-1, 0])
        nxt = (pos[0] + int(delta[0]), pos[1] + int(delta[1]))
        bad_row = nxt[0] < 0 or nxt[0] >= self.height
        bad_col = nxt[1] < 0 or nxt[1] >= self.width
        if bad_row or bad_col or nxt in self.walls:
            nxt = pos
        return nxt

    def step(self, state, action, local_rng):
        pos = self.index_state(int(state))
        nxt = self.move(pos, action, local_rng)
        reward = self.step_cost
        done = False
        if nxt in self.traps:
            reward = reward + float(self.traps[nxt])
        if nxt == self.goal:
            reward = reward + self.goal_reward
            done = True
        return self.state_index(nxt), reward, done


def make_rl_ladder(continuous=False):
    envs = []
    envs.append(LadderEnv("D1 two-state chain", 1, 2, (0, 0), (0, 1), tuple(), 0.0, 0.0, 0.0, 1.0, {}, 4, continuous))
    envs.append(LadderEnv("D2 slippery 3-state", 1, 3, (0, 0), (0, 2), tuple(), 0.15, 0.0, -0.01, 1.0, {}, 8, continuous))
    envs.append(LadderEnv("D3 4x4 gridworld", 4, 4, (3, 0), (0, 3), ((1, 1),), 0.05, 0.0, -0.02, 1.0, {(2, 2): -0.25}, 24, continuous))
    envs.append(LadderEnv("D4 windy stochastic grid", 5, 5, (4, 0), (0, 4), ((1, 1), (2, 1), (3, 3)), 0.12, 0.18, -0.025, 1.1, {(2, 3): -0.4}, 35, continuous))
    envs.append(LadderEnv("D5 sparse reward grid", 6, 6, (5, 0), (0, 5), ((1, 1), (1, 2), (2, 2), (3, 4), (4, 1)), 0.18, 0.20, -0.03, 1.5, {(2, 4): -0.6, (4, 4): -0.3}, 50, continuous))
    return envs


def softmax(logits):
    shifted = logits - np.max(logits, axis=-1, keepdims=True)
    exp_logits = np.exp(shifted)
    return exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)


def discounted_returns(rewards, gamma):
    returns = np.zeros(len(rewards), dtype=float)
    running = 0.0
    for t in range(len(rewards) - 1, -1, -1):
        running = float(rewards[t]) + gamma * running
        returns[t] = running
    return returns


def rollout(env, logits, local_rng, gamma=0.9):
    states = []
    actions = []
    rewards = []
    state = env.reset()
    for _ in range(env.max_steps):
        probs = softmax(logits[state])
        action = int(local_rng.choice(env.n_actions, p=probs))
        next_state, reward, done = env.step(state, action, local_rng)
        states.append(state)
        actions.append(action)
        rewards.append(reward)
        state = next_state
        if done:
            break
    returns = discounted_returns(np.array(rewards), gamma)
    return np.array(states), np.array(actions), np.array(rewards), returns


def evaluate_policy(env, logits, episodes=20, gamma=0.9):
    values = []
    local_rng = np.random.default_rng(SEED + env.n_states)
    for _ in range(episodes):
        _, _, rewards, _ = rollout(env, logits, local_rng, gamma)
        values.append(float(np.sum(rewards)))
    return float(np.mean(values))


def value_iteration(env, gamma=0.9, iterations=120):
    values = np.zeros(env.n_states)
    local_rng = np.random.default_rng(SEED)
    for _ in range(iterations):
        new_values = values.copy()
        for state in range(env.n_states):
            pos = env.index_state(state)
            if pos == env.goal:
                continue
            q_values = []
            for action in range(env.n_actions):
                next_state, reward, done = env.step(state, action, local_rng)
                q_values.append(reward + gamma * values[next_state] * (1.0 - float(done)))
            new_values[state] = np.max(q_values)
        values = new_values
    return values


def greedy_logits_from_values(env, values, gamma=0.9, scale=5.0):
    logits = np.zeros((env.n_states, env.n_actions))
    local_rng = np.random.default_rng(SEED + 7)
    for state in range(env.n_states):
        for action in range(env.n_actions):
            next_state, reward, done = env.step(state, action, local_rng)
            logits[state, action] = scale * (reward + gamma * values[next_state] * (1.0 - float(done)))
    return logits


def plot_policy_panel(ax, env, values, logits, title):
    grid = values.reshape(env.height, env.width)
    ax.imshow(grid, cmap="viridis")
    probs = softmax(logits)
    for state in range(env.n_states):
        row, col = env.index_state(state)
        if (row, col) in env.walls:
            ax.text(col, row, "#", ha="center", va="center", color="white")
            continue
        best = int(np.argmax(probs[state]))
        ax.text(col, row, ACTION_NAMES[best], ha="center", va="center", color="white")
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])


def summarize_ladder(envs):
    for env in envs:
        print(env.name, "states", env.n_states, "actions", env.n_actions, "slip", env.slip, "wind", env.wind)
        print("start", env.start, "goal", env.goal, "walls", len(env.walls), "traps", env.traps)


def train_reinforce(env, episodes=80, gamma=0.9, lr=0.08, baseline=True):
    logits = np.zeros((env.n_states, env.n_actions))
    value = np.zeros(env.n_states)
    curve = []
    local_rng = np.random.default_rng(SEED + env.n_states)
    for _ in range(episodes):
        states, actions, rewards, returns = rollout(env, logits, local_rng, gamma)
        if len(states) == 0:
            continue
        advantages = returns.copy()
        if baseline:
            advantages = returns - value[states]
            for state, target in zip(states, returns):
                value[state] = value[state] + 0.15 * (target - value[state])
        for state, action, advantage in zip(states, actions, advantages):
            probs = softmax(logits[state])
            grad = -probs
            grad[action] = grad[action] + 1.0
            logits[state] = logits[state] + lr * advantage * grad
        curve.append(float(np.sum(rewards)))
    return logits, value, np.array(curve)


def train_actor_critic(env, episodes=80, gamma=0.9, actor_lr=0.05, critic_lr=0.12, normalize=True):
    logits = np.zeros((env.n_states, env.n_actions))
    value = np.zeros(env.n_states)
    curve = []
    local_rng = np.random.default_rng(SEED + 2 * env.n_states)
    for _ in range(episodes):
        states = []
        actions = []
        deltas = []
        rewards = []
        state = env.reset()
        for _ in range(env.max_steps):
            probs = softmax(logits[state])
            action = int(local_rng.choice(env.n_actions, p=probs))
            next_state, reward, done = env.step(state, action, local_rng)
            target = reward + gamma * value[next_state] * (1.0 - float(done))
            delta = target - value[state]
            value[state] = value[state] + critic_lr * delta
            states.append(state)
            actions.append(action)
            deltas.append(delta)
            rewards.append(reward)
            state = next_state
            if done:
                break
        advantages = np.array(deltas)
        if normalize and len(advantages) > 1:
            advantages = (advantages - np.mean(advantages)) / (np.std(advantages) + 1e-8)
        for state, action, advantage in zip(states, actions, advantages):
            probs = softmax(logits[state])
            grad = -probs
            grad[action] = grad[action] + 1.0
            logits[state] = logits[state] + actor_lr * advantage * grad
        curve.append(float(np.sum(rewards)))
    return logits, value, np.array(curve)


def gae_advantages(rewards, values, next_values, dones, gamma=0.9, lam=0.95):
    deltas = rewards + gamma * next_values * (1.0 - dones) - values
    adv = np.zeros_like(rewards, dtype=float)
    running = 0.0
    for t in range(len(rewards) - 1, -1, -1):
        running = deltas[t] + gamma * lam * (1.0 - dones[t]) * running
        adv[t] = running
    return deltas, adv


def train_gae_policy(env, lam=0.95, episodes=80, gamma=0.9):
    logits = np.zeros((env.n_states, env.n_actions))
    value = np.zeros(env.n_states)
    curve = []
    local_rng = np.random.default_rng(SEED + int(100 * lam) + env.n_states)
    for _ in range(episodes):
        states, actions, rewards, returns = rollout(env, logits, local_rng, gamma)
        if len(states) == 0:
            continue
        next_values = np.zeros(len(states))
        dones = np.zeros(len(states))
        for i, state in enumerate(states):
            if i + 1 < len(states):
                next_values[i] = value[states[i + 1]]
            else:
                dones[i] = 1.0
        _, advantages = gae_advantages(rewards, value[states], next_values, dones, gamma, lam)
        for state, target in zip(states, returns):
            value[state] = value[state] + 0.12 * (target - value[state])
        if len(advantages) > 1:
            advantages = (advantages - np.mean(advantages)) / (np.std(advantages) + 1e-8)
        for state, action, advantage in zip(states, actions, advantages):
            probs = softmax(logits[state])
            grad = -probs
            grad[action] = grad[action] + 1.0
            logits[state] = logits[state] + 0.05 * advantage * grad
        curve.append(float(np.sum(rewards)))
    return logits, value, np.array(curve)


def ppo_surrogate(old_probs, new_probs, actions, advantages, clip=0.2):
    chosen_old = old_probs[np.arange(len(actions)), actions]
    chosen_new = new_probs[np.arange(len(actions)), actions]
    ratios = chosen_new / np.maximum(chosen_old, 1e-8)
    unclipped = ratios * advantages
    clipped = np.clip(ratios, 1.0 - clip, 1.0 + clip) * advantages
    return ratios, np.minimum(unclipped, clipped)


def train_ppo(env, episodes=80, gamma=0.9, clip=0.2, clipped=True):
    logits = np.zeros((env.n_states, env.n_actions))
    value = np.zeros(env.n_states)
    curve = []
    local_rng = np.random.default_rng(SEED + 3 * env.n_states)
    for _ in range(episodes):
        states, actions, rewards, returns = rollout(env, logits, local_rng, gamma)
        if len(states) == 0:
            continue
        old_probs = softmax(logits[states])
        advantages = returns - value[states]
        if len(advantages) > 1:
            advantages = (advantages - np.mean(advantages)) / (np.std(advantages) + 1e-8)
        for _epoch in range(3):
            new_probs = softmax(logits[states])
            ratios, weights = ppo_surrogate(old_probs, new_probs, actions, advantages, clip)
            if not clipped:
                weights = ratios * advantages
            for state, action, weight in zip(states, actions, weights):
                probs = softmax(logits[state])
                grad = -probs
                grad[action] = grad[action] + 1.0
                logits[state] = logits[state] + 0.04 * weight * grad
        for state, target in zip(states, returns):
            value[state] = value[state] + 0.15 * (target - value[state])
        curve.append(float(np.sum(rewards)))
    return logits, value, np.array(curve)


def distributional_backup(rewards, gamma):
    atoms = discounted_returns(np.array(rewards, dtype=float), gamma)
    probs = np.ones_like(atoms) / len(atoms)
    return atoms, probs, float(atoms[0])


def train_distributional(env, atoms=21, episodes=60, gamma=0.9):
    support = np.linspace(-1.0, 2.0, atoms)
    pmf = np.ones((env.n_states, env.n_actions, atoms)) / atoms
    logits = np.zeros((env.n_states, env.n_actions))
    curve = []
    errors = []
    optimal = value_iteration(env, gamma)
    local_rng = np.random.default_rng(SEED + 4 * env.n_states)
    for _ in range(episodes):
        state = env.reset()
        episode_reward = 0.0
        for _step in range(env.max_steps):
            means = np.sum(pmf[state] * support[None, :], axis=1)
            action = int(np.argmax(means + local_rng.normal(0.0, 0.03, env.n_actions)))
            next_state, reward, done = env.step(state, action, local_rng)
            next_action = int(np.argmax(np.sum(pmf[next_state] * support[None, :], axis=1)))
            shifted = reward + gamma * support * (1.0 - float(done))
            target = np.interp(support, shifted, pmf[next_state, next_action], left=0.0, right=0.0)
            if np.sum(target) <= 0.0:
                nearest = int(np.argmin(np.abs(support - reward)))
                target = np.zeros(atoms)
                target[nearest] = 1.0
            target = target / np.sum(target)
            pmf[state, action] = 0.9 * pmf[state, action] + 0.1 * target
            episode_reward = episode_reward + reward
            state = next_state
            if done:
                break
        learned = np.max(np.sum(pmf * support[None, None, :], axis=2), axis=1)
        errors.append(float(np.mean(np.abs(learned - optimal))))
        curve.append(episode_reward)
    values = np.max(np.sum(pmf * support[None, None, :], axis=2), axis=1)
    logits = greedy_logits_from_values(env, values, gamma)
    spread = np.mean(np.std(pmf * support[None, None, :], axis=2))
    return logits, values, np.array(errors), float(spread)


def continuous_features(state, action):
    return np.array([1.0, state, action, state * action, action * action])


def deterministic_actor_critic(reward, next_q, gamma):
    target = reward + gamma * next_q
    critic_prediction = 0.4
    critic_error = target - critic_prediction
    return target, critic_error


def td3_toy_update(state, reward, next_state, actor_w, q1_w, q2_w, gamma=0.9, noise=0.05):
    action = float(np.tanh(actor_w * state))
    next_action = float(np.clip(np.tanh(actor_w * next_state) + noise, -1.0, 1.0))
    q1_next = float(continuous_features(next_state, next_action) @ q1_w)
    q2_next = float(continuous_features(next_state, next_action) @ q2_w)
    target = reward + gamma * min(q1_next, q2_next)
    prediction = float(continuous_features(state, action) @ q1_w)
    td_error = target - prediction
    q1_w = q1_w + 0.05 * td_error * continuous_features(state, action)
    return action, target, td_error, q1_w


def train_td3_ladder(env, episodes=70, gamma=0.9, twin=True):
    actor_w = 0.2
    q1_w = np.array([0.0, 0.2, 0.1, 0.0, -0.05])
    q2_w = np.array([-0.02, 0.15, 0.08, 0.0, -0.08])
    curve = []
    local_rng = np.random.default_rng(SEED + 5 * env.n_states)
    for _ in range(episodes):
        state = 0.0
        total = 0.0
        for _step in range(env.max_steps):
            action = float(np.clip(np.tanh(actor_w * state) + local_rng.normal(0.0, 0.15), -1.0, 1.0))
            target_position = 1.0
            next_state = float(np.clip(state + 0.25 * action + local_rng.normal(0.0, 0.03), -1.2, 1.2))
            reward = -abs(target_position - next_state) - 0.05 * action * action
            next_action = float(np.clip(np.tanh(actor_w * next_state) + local_rng.normal(0.0, 0.05), -1.0, 1.0))
            q1_next = float(continuous_features(next_state, next_action) @ q1_w)
            q2_next = float(continuous_features(next_state, next_action) @ q2_w)
            next_value = min(q1_next, q2_next) if twin else q1_next
            target = reward + gamma * next_value
            feat = continuous_features(state, action)
            td_error_1 = target - float(feat @ q1_w)
            td_error_2 = target - float(feat @ q2_w)
            q1_w = q1_w + 0.03 * td_error_1 * feat
            q2_w = q2_w + 0.03 * td_error_2 * feat
            actor_grad = q1_w[2] + q1_w[3] * state + 2.0 * q1_w[4] * np.tanh(actor_w * state)
            actor_w = actor_w + 0.01 * actor_grad * state
            total = total + reward
            state = next_state
        curve.append(total)
    values = np.linspace(-1.0, 1.0, env.n_states)
    logits = np.tile(np.array([-actor_w, actor_w, -0.5 * actor_w, 0.5 * actor_w]), (env.n_states, 1))
    return logits, values, np.array(curve)


## The concept, built once: REINFORCE
The lesson formula is $$
abla_\theta J(\theta)=\mathbb{E}[
abla_\theta\log\pi_\theta(a_t\mid s_t)A_t]$$. With logits $[1,0]$, softmax gives $\pi(a_0)=0.731$ and $\pi(a_1)=0.269$; with rewards $[2,0]$, expected reward is $1.462$. The D1 return remains $2.620$.

In [ ]:

def reinforce_step(logits, action, advantage, lr=0.1):
    probs = softmax(logits)
    grad = -probs
    grad[action] = grad[action] + 1.0
    updated = logits + lr * advantage * grad
    return probs, grad, updated

logits = np.array([1.0, 0.0])
rewards = np.array([1.0, 0.0, 2.0])
advantage = float(discounted_returns(rewards, 0.9)[0])
probs, grad, updated = reinforce_step(logits, 0, advantage)
expected_reward = float(probs @ np.array([2.0, 0.0]))

print("probabilities", probs)
print("gradient", grad)
print("updated logits", updated)
print("expected reward", expected_reward)

assert round(probs[0], 3) == 0.731
assert round(probs[1], 3) == 0.269
assert round(expected_reward, 3) == 1.462
assert round(advantage, 3) == 2.620


A complete episode return weights every log-probability in REINFORCE. A baseline may be subtracted without changing the expected gradient, so the same method can reduce variance while keeping the direct policy optimization objective.

In [ ]:

baseline = 1.0
centered_advantage = advantage - baseline
_, centered_grad, centered_update = reinforce_step(logits, 0, centered_advantage)

print("centered advantage", centered_advantage)
print("centered update", centered_update)
assert centered_advantage < advantage
assert centered_update[0] > logits[0]


## The dataset ladder: F12 sequential-decision environments
We build the D1-D5 ladder inline, from a two-state chain to a sparse stochastic windy grid. Every rung has a start, goal, transition noise, and a small enough state space for CPU-only NumPy experiments.

In [ ]:

envs = make_rl_ladder()
summarize_ladder(envs)

for env in envs:
    sample_state = env.reset()
    sample_next, sample_reward, sample_done = env.step(sample_state, 3, np.random.default_rng(SEED))
    print(env.name, "sample", sample_state, "->", sample_next, "reward", round(sample_reward, 3), "done", sample_done)


## Run REINFORCE across D1-D5
The same softmax policy and episode-return update is used on every rung. The metric is average return.

In [ ]:

results = []
artifacts = []
for env in envs:
    logits, values, curve = train_reinforce(env, baseline=True)
    random_logits = np.zeros_like(logits)
    learned_return = evaluate_policy(env, logits)
    random_return = evaluate_policy(env, random_logits)
    artifacts.append((env, logits, values, curve))
    results.append((env.name, learned_return, random_return))

print("rung | learned_return | random_return")
for name, learned_return, random_return in results:
    print(name, round(learned_return, 3), round(random_return, 3))


## Results visualization
The panels show the learned value/policy and the learning curve for each rung.

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, (env, logits, values, curve) in zip(axes[0], artifacts):
    plot_policy_panel(ax, env, values, logits, env.name)
for ax, (env, logits, values, curve) in zip(axes[1], artifacts):
    ax.plot(curve, color="tab:green")
    ax.set_title("return " + env.name.split()[0])
    ax.set_xlabel("episode")
    ax.set_ylabel("return")
plt.tight_layout()


## Pitfall on D5: reward is not return
Weighting actions only by immediate reward ignores delayed consequences. The fix uses full episode return and a baseline.

In [ ]:

d5 = envs[-1]
wrong_logits, wrong_values, wrong_curve = train_reinforce(d5, baseline=False, gamma=0.0)
fixed_logits, fixed_values, fixed_curve = train_reinforce(d5, baseline=True, gamma=0.9)
wrong_return = evaluate_policy(d5, wrong_logits)
fixed_return = evaluate_policy(d5, fixed_logits)

print("immediate-reward policy return", round(wrong_return, 3))
print("full-return baseline policy return", round(fixed_return, 3))
assert fixed_return >= wrong_return - 0.5


## Evaluate it + practice
- Metric: compare return or advantage/value error against a no-skill random-policy baseline on every rung.
- Sanity check: D1 should solve first because it has the shortest horizon and no stochastic transition.
- Ablation: turn off the key stabilizer for this lesson and the D5 metric should drop or become noisier.
- Failure signals: exploding logits, a value table with impossible magnitudes, or a policy that never reaches the goal.

Practice prompts:
1. Change $\gamma$ from 0.9 to 0.7 and explain which rungs lose the most return.
2. Remove the baseline and compare the standard deviation of episode returns.
3. Increase the softmax temperature and explain how exploration changes.


In [ ]:
# Your code here


In [ ]:
# Your code here
